In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/structured_piano_v3
!mkdir -p /content/structured_piano_v3

!unzip -oq \
  "/content/drive/MyDrive/colab_workspace/data/structured_piano_v3_token_data.zip" \
  -d /content/structured_piano_v3

!find /content/structured_piano_v3/structured_token_data -name "*.npy" | wc -l
!head -n 3 /content/structured_piano_v3/structured_token_manifest.csv
!cat /content/structured_piano_v3/structured_tokenizer_config.json

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

PROJECT_DIR = Path("/content/structured_piano_v3")
TOKEN_DIR = PROJECT_DIR / "structured_token_data"
MANIFEST_PATH = PROJECT_DIR / "structured_token_manifest.csv"
CONFIG_PATH = PROJECT_DIR / "structured_tokenizer_config.json"

with open(CONFIG_PATH, encoding="utf-8") as f:
    tokenizer_config = json.load(f)

VOCAB_SIZE = tokenizer_config["vocab_size"]  # 229
CONTEXT_LENGTH = 4096
BATCH_SIZE = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

PAD_TOKEN = 0
BOS_TOKEN = 1
EOS_TOKEN = 2
BAR_TOKEN = 3
CHORD_START = 4
NUM_CHORD_TOKENS = 25
POSITION_START = CHORD_START + NUM_CHORD_TOKENS
VELOCITY_START = POSITION_START + 16
PITCH_START = VELOCITY_START + 32
DURATION_START = PITCH_START + 88

class StructuredV3Dataset(Dataset):
    def __init__(self, dataframe, start_probability=0.30):
        self.df = dataframe.reset_index(drop=True)
        self.start_probability = start_probability

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        token_path = TOKEN_DIR / Path(row["token_path"]).name
        tokens = np.load(token_path).astype(np.int64)

        max_start = max(0, len(tokens) - CONTEXT_LENGTH - 1)

        if max_start == 0 or np.random.random() < self.start_probability:
            start = 0
        else:
            start = np.random.randint(0, max_start + 1)

        chunk = tokens[start : start + CONTEXT_LENGTH + 1]

        if len(chunk) < CONTEXT_LENGTH + 1:
            chunk = np.pad(
                chunk,
                (0, CONTEXT_LENGTH + 1 - len(chunk)),
                constant_values=PAD_TOKEN,
            )

        x = torch.from_numpy(chunk[:-1])
        y = torch.from_numpy(chunk[1:])

        return x, y, torch.tensor(int(row["composer_id"]))

manifest = pd.read_csv(MANIFEST_PATH)

train_df = manifest[manifest["split"] == "train"].copy()
val_df = manifest[manifest["split"] == "validation"].copy()

train_dataset = StructuredV3Dataset(train_df, start_probability=0.30)
val_dataset = StructuredV3Dataset(val_df, start_probability=0.30)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

x, y, composer_ids = next(iter(train_loader))

print("V3 vocab:", VOCAB_SIZE)
print("# of training songs:", len(train_dataset))
print("# of validation songs:", len(val_dataset))
print("Input shape:", x.shape)
print("Output shape:", y.shape)
print("Composer ID:", composer_ids.tolist())
print("Token range:", x.min().item(), "~", x.max().item())

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 8
FFN_DIM = 2048
DROPOUT = 0.10
NUM_COMPOSERS = 6


def apply_rope(x):
    """
    x: (batch, heads, time, head_dim)
    """
    _, _, length, head_dim = x.shape
    device = x.device

    positions = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )

    frequencies = torch.exp(
        -math.log(10000.0)
        * torch.arange(0, head_dim, 2, device=device).float()
        / head_dim
    )

    angles = positions[:, None] * frequencies[None, :]
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_even * sin + x_odd * cos

    return torch.stack(
        (rotated_even, rotated_odd),
        dim=-1,
    ).flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        batch, length, channels = x.shape

        q, k, v = self.qkv(x).chunk(3, dim=-1)

        q = q.view(
            batch, length, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch, length, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch, length, self.num_heads, self.head_dim
        ).transpose(1, 2)

        q = apply_rope(q)
        k = apply_rope(k)

        x = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        x = x.transpose(1, 2).contiguous().view(
            batch, length, channels
        )

        return self.out(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(
            d_model,
            num_heads,
            dropout,
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ComposerMusicTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_composers,
        d_model,
        num_heads,
        num_layers,
        ffn_dim,
        dropout,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_TOKEN,
        )

        self.composer_embedding = nn.Embedding(
            num_composers,
            d_model,
        )

        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                num_heads,
                ffn_dim,
                dropout,
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tokens, composer_ids):
        x = self.token_embedding(tokens)

        composer = self.composer_embedding(composer_ids)[:, None, :]
        x = self.dropout(x + composer)

        for block in self.blocks:
            x = block(x)

        return self.lm_head(self.norm(x))


def init_music_transformer(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

        if module.bias is not None:
            nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)


v3_model = ComposerMusicTransformer(
    vocab_size=VOCAB_SIZE,
    num_composers=NUM_COMPOSERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ffn_dim=FFN_DIM,
    dropout=DROPOUT,
).to(device)

v3_model.apply(init_music_transformer)

v3_model.lm_head.weight = v3_model.token_embedding.weight

with torch.no_grad():
    v3_model.token_embedding.weight[PAD_TOKEN].zero_()

with torch.no_grad():
    test_logits = v3_model(
        x.to(device),
        composer_ids.to(device),
    )

print(
    "parameter:",
    f"{sum(p.numel() for p in v3_model.parameters()) / 1e6:.2f} M",
)
print("logits shape:", test_logits.shape)
print(
    "expected shape:",
    (BATCH_SIZE, CONTEXT_LENGTH, VOCAB_SIZE),
)

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F


LEARNING_RATE = 3e-4
EPOCHS = 20

optimizer = torch.optim.AdamW(
    v3_model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler("cuda")

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/colab_workspace/models/"
    "structured_v3_piano_transformer"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = CHECKPOINT_DIR / "best_structured_v3_piano_transformer.pt"
LAST_PATH = CHECKPOINT_DIR / "last_structured_v3_piano_transformer.pt"

composer_map = {
    "Frédéric Chopin": 0,
    "Franz Schubert": 1,
    "Ludwig van Beethoven": 2,
    "Johann Sebastian Bach": 3,
    "Franz Liszt": 4,
    "OTHER": 5,
}

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    v3_model.train()
    train_loss_sum = 0.0

    for batch_x, batch_y, batch_composer_ids in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        batch_composer_ids = batch_composer_ids.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = v3_model(batch_x, batch_composer_ids)

            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                batch_y.reshape(-1),
                ignore_index=PAD_TOKEN,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            v3_model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item()

    train_loss = train_loss_sum / len(train_loader)

    v3_model.eval()
    val_loss_sum = 0.0

    with torch.inference_mode():
        for batch_x, batch_y, batch_composer_ids in val_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            batch_composer_ids = batch_composer_ids.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = v3_model(batch_x, batch_composer_ids)

                loss = F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE),
                    batch_y.reshape(-1),
                    ignore_index=PAD_TOKEN,
                )

            val_loss_sum += loss.item()

    val_loss = val_loss_sum / len(val_loader)
    is_best = val_loss < best_val_loss

    if is_best:
        best_val_loss = val_loss

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": v3_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "vocab_size": VOCAB_SIZE,
        "num_composers": NUM_COMPOSERS,
        "context_length": CONTEXT_LENGTH,
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "ffn_dim": FFN_DIM,
        "dropout": DROPOUT,
        "tokenizer_format": tokenizer_config["format"],
        "composer_map": composer_map,
    }

    torch.save(checkpoint, LAST_PATH)

    if is_best:
        torch.save(checkpoint, BEST_PATH)

    print(
        f"V3 Epoch {epoch:02d}/{EPOCHS} | "
        f"train: {train_loss:.4f} | "
        f"validation: {val_loss:.4f} | "
        f"best: {best_val_loss:.4f}"
    )

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class TransposedStructuredV3Dataset(Dataset):
    def __init__(self, dataframe, start_probability=0.30, max_transpose=0):
        self.df = dataframe.reset_index(drop=True)
        self.start_probability = start_probability
        self.max_transpose = max_transpose

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        token_path = TOKEN_DIR / Path(row["token_path"]).name
        tokens = np.load(token_path).astype(np.int64)

        max_start = max(0, len(tokens) - CONTEXT_LENGTH - 1)

        if max_start == 0 or np.random.random() < self.start_probability:
            start = 0
        else:
            bar_starts = np.flatnonzero(tokens == BAR_TOKEN)
            valid_starts = bar_starts[bar_starts <= max_start]

            if len(valid_starts) > 0:
                start = int(np.random.choice(valid_starts))
            else:
                start = np.random.randint(0, max_start + 1)

        chunk = tokens[start : start + CONTEXT_LENGTH + 1].copy()
        if self.max_transpose > 0:
            pitch_mask = (
                (chunk >= PITCH_START)
                & (chunk < DURATION_START)
            )

            if np.any(pitch_mask):
                pitch_indices = chunk[pitch_mask] - PITCH_START

                min_shift = max(
                    -self.max_transpose,
                    -int(pitch_indices.min()),
                )
                max_shift = min(
                    self.max_transpose,
                    87 - int(pitch_indices.max()),
                )

                shift = random.randint(min_shift, max_shift)

                chunk[pitch_mask] += shift

                chord_mask = (
                    (chunk >= CHORD_START)
                    & (chunk < CHORD_START + 24)
                )

                chord_ids = chunk[chord_mask] - CHORD_START
                qualities = chord_ids // 12
                roots = chord_ids % 12

                chunk[chord_mask] = (
                    CHORD_START
                    + qualities * 12
                    + (roots + shift) % 12
                )

        if len(chunk) < CONTEXT_LENGTH + 1:
            chunk = np.pad(
                chunk,
                (0, CONTEXT_LENGTH + 1 - len(chunk)),
                constant_values=PAD_TOKEN,
            )

        x = torch.from_numpy(chunk[:-1])
        y = torch.from_numpy(chunk[1:])

        return x, y, torch.tensor(int(row["composer_id"]))

train_dataset_aug = TransposedStructuredV3Dataset(
    train_df,
    start_probability=0.30,
    max_transpose=5,
)

val_dataset_v3 = TransposedStructuredV3Dataset(
    val_df,
    start_probability=0.30,
    max_transpose=0,
)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

val_loader_v3 = DataLoader(
    val_dataset_v3,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

base_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False,
)

v3_model.load_state_dict(base_checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    v3_model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

optimizer.load_state_dict(
    base_checkpoint["optimizer_state_dict"]
)

for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)

for group in optimizer.param_groups:
    group["lr"] = 1e-4

scaler = torch.amp.GradScaler("cuda")
scaler.load_state_dict(base_checkpoint["scaler_state_dict"])

AUG_BEST_PATH = (
    CHECKPOINT_DIR
    / "best_structured_v3_piano_transformer_augmented.pt"
)

AUG_LAST_PATH = (
    CHECKPOINT_DIR
    / "last_structured_v3_piano_transformer_augmented.pt"
)

best_val_loss = base_checkpoint["best_val_loss"]
START_EPOCH = base_checkpoint["epoch"] + 1
TARGET_EPOCH = 34
print(
    f"From epoch {START_EPOCH} to {TARGET_EPOCH}"
)
print(f"Starting best validation loss: {best_val_loss:.4f}")

for epoch in range(START_EPOCH, TARGET_EPOCH + 1):
    v3_model.train()
    train_loss_sum = 0.0

    for batch_x, batch_y, batch_composer_ids in train_loader_aug:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        batch_composer_ids = batch_composer_ids.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = v3_model(batch_x, batch_composer_ids)

            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                batch_y.reshape(-1),
                ignore_index=PAD_TOKEN,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            v3_model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item()

    train_loss = train_loss_sum / len(train_loader_aug)

    v3_model.eval()
    val_loss_sum = 0.0

    with torch.inference_mode():
        for batch_x, batch_y, batch_composer_ids in val_loader_v3:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            batch_composer_ids = batch_composer_ids.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = v3_model(batch_x, batch_composer_ids)

                loss = F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE),
                    batch_y.reshape(-1),
                    ignore_index=PAD_TOKEN,
                )

            val_loss_sum += loss.item()

    val_loss = val_loss_sum / len(val_loader_v3)
    is_best = val_loss < best_val_loss

    if is_best:
        best_val_loss = val_loss

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": v3_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "vocab_size": VOCAB_SIZE,
        "num_composers": NUM_COMPOSERS,
        "context_length": CONTEXT_LENGTH,
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "ffn_dim": FFN_DIM,
        "dropout": DROPOUT,
        "tokenizer_format": tokenizer_config["format"],
        "composer_map": composer_map,
        "augmentation": "pitch_transpose_plus_matching_chord_root",
    }

    torch.save(checkpoint, AUG_LAST_PATH)

    if is_best:
        torch.save(checkpoint, AUG_BEST_PATH)

    print(
        f"V3 augmented {epoch:02d}/{TARGET_EPOCH} | "
        f"train: {train_loss:.4f} | "
        f"validation: {val_loss:.4f} | "
        f"best: {best_val_loss:.4f}"
    )